# Zeek Log Cleaning (zat-based)

This notebook reads common Zeek log types using **zat**, applies light normalization + stable pseudonymization (reusing `mappings/`), and writes cleaned outputs to `data/cleaned_zeek/`.

Supported log types:
- `conn.log`
- `dns.log`
- `http.log`
- `ssl.log`
- `files.log`
- `ssh.log`


In [1]:
# --- 0) Install & import dependencies (zat) ---
import sys, subprocess
import os, json, hashlib, re
from pathlib import Path
import pandas as pd
from zat.log_to_dataframe import LogToDataFrame
from collections import defaultdict

pd.set_option("display.max_columns", 200)


In [2]:
# --- 1) Paths & log list ---
PROJECT_ROOT = Path.cwd().parent.resolve()

DATA_DIR = PROJECT_ROOT / "data"         # put *.log here
OUT_DIR = PROJECT_ROOT / "sanidata"
MAP_DIR = PROJECT_ROOT / "mappings"    # reuse existing mapping dir

OUT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

# --- 1) Zeek log "types" we care about (filename-based) ---
ZEEK_FILES = {
    "conn":  "conn.log",
    "dns":   "dns.log",
    "http":  "http.log",
    "ssl":   "ssl.log",
    "files": "files.log",
    "ssh":   "ssh.log",
}

def extract_scenario(path: Path) -> str:
    """
    Extract scenario id like 'sc1' from any part of the path.
    Falls back to 'unknown' if not found.
    """
    for part in path.parts:
        if part.startswith("sc") and part[2:].isdigit():
            return part
    return "unknown"

def collect_inputs(base: Path, files: dict) -> dict:
    """
    Return:
      inputs[scenario][log_type] -> list[Path]  (1 or many)
    """
    inputs = defaultdict(lambda: defaultdict(list))
    for log_type, fname in files.items():
        for p in base.rglob(fname):
            sc = extract_scenario(p)
            inputs[sc][log_type].append(p)
    return inputs

INPUTS = collect_inputs(DATA_DIR, ZEEK_FILES)

# Quick sanity print: how many matches per scenario/log_type
for sc in sorted(INPUTS):
    counts = {t: len(ps) for t, ps in INPUTS[sc].items()}
    print(f"{sc}: {counts}")


print("Input dir:", DATA_DIR)
print("Output dir:", OUT_DIR)
print("Mappings dir:", MAP_DIR)


sc3: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
sc4: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
sc7: {'conn': 1, 'dns': 1, 'http': 1, 'ssl': 1, 'files': 1, 'ssh': 1}
Input dir: /Users/zhuoran/Projects/SSCMDataset/data
Output dir: /Users/zhuoran/Projects/SSCMDataset/sanidata
Mappings dir: /Users/zhuoran/Projects/SSCMDataset/mappings


In [3]:
# --- 2) Stable SALT (reuse same pattern) ---
SALT_PATH = MAP_DIR / "salt.txt"
if SALT_PATH.exists():
    SALT = SALT_PATH.read_text().strip()
else:
    SALT = hashlib.sha256(os.urandom(32)).hexdigest()
    SALT_PATH.write_text(SALT)


In [4]:

def _load_map(name: str) -> dict:
    p = MAP_DIR / f"{name}.json"
    if p.exists():
        return json.loads(p.read_text())
    return {}

def _save_map(name: str, m: dict):
    p = MAP_DIR / f"{name}.json"
    p.write_text(json.dumps(m, ensure_ascii=False, indent=2))

def stable_hash(text: str) -> str:
    return hashlib.sha256((SALT + str(text)).encode("utf-8")).hexdigest()

def map_token(value, prefix: str, map_name: str, digits: int = 6):
    """Map a raw value to a stable pseudonym like ip_012345, user_000123, etc.
    Uses mappings/<map_name>.json for persistence across runs.
    """
    m = _load_map(map_name)
    key = str(value)
    if key in m:
        return m[key]

    h = stable_hash(key)
    mod = 10 ** digits
    n = int(h[:16], 16) % mod
    token = f'{prefix}_{n:0{digits}d}'

    # Resolve rare collisions: if token already used for a different key, walk forward
    used = set(m.values())
    while token in used:
        n = (n + 1) % mod
        token = f'{prefix}_{n:0{digits}d}'

    m[key] = token
    _save_map(map_name, m)
    return token

def normalize_scalar(x):
    """Normalize common Zeek placeholders."""
    if x is None:
        return None
    if isinstance(x, float) and pd.isna(x):
        return None
    if isinstance(x, str):
        s = x.strip()
        if s in {"-", ""}:
            return None
        if s.lower() in {"(empty)", "null", "none", "nan"}:
            return None
        return s
    return x


In [5]:
# --- 3) Sanitization policy (newt-only user; Azure-only FQDN) ---
# Only pseudonymize this username (case-insensitive). Everything else is preserved.
ONLY_PSEUDONYMIZE_USERS = {'newt'}

# Azure/Azure-hosted suffix allowlist (acts like *.suffix wildcard)
AZURE_FQDN_SUFFIXES = (
    'trafficmanager.net',
    'azure.com',
    'azure.net',
    'azureedge.net',
    'cloudapp.azure.com',
    'azurewebsites.net',
    'windows.net',
    'core.windows.net',
    'database.windows.net',
    'redis.cache.windows.net',
    'servicebus.windows.net',
    'vault.azure.net',
    'azure-api.net',
)

def _is_empty(v) -> bool:
    return v is None or (isinstance(v, float) and pd.isna(v)) or (isinstance(v, str) and v.strip() == '')

def is_azure_fqdn(fqdn: str) -> bool:
    if _is_empty(fqdn):
        return False
    f = str(fqdn).strip().lower().rstrip('.')
    return any(f == sfx or f.endswith('.' + sfx) for sfx in AZURE_FQDN_SUFFIXES)

def sanitize_fqdn(fqdn: str) -> str:
    """Pseudonymize Azure/Azure-hosted FQDNs only; keep all other FQDNs unchanged."""
    if _is_empty(fqdn) or pd.isna(fqdn):
        return fqdn
    s = str(fqdn).strip()
    base = s.rstrip('.')
    if not is_azure_fqdn(base):
        return s
    tok = map_token(base.lower(), 'FQDN', 'fqdn_map', digits=4)
    # keep a plausible domain shape without leaking the original
    return f'{tok}.example'

def sanitize_user_token(user: str) -> str:
    """Pseudonymize only allowlisted user tokens (case-insensitive)."""
    if _is_empty(user) or pd.isna(user):
        return user
    u = str(user).strip()
    # DOMAIN\\USER -> only consider USER for allowlist
    if '\\' in u:
        dom, name = u.rsplit('\\', 1)
        name_norm = name.strip().lower()
        if name_norm in ONLY_PSEUDONYMIZE_USERS:
            return dom + '\\' + map_token(name_norm, 'USER', 'user_map', digits=3)
        return u
    name_norm = u.lower()
    if name_norm in ONLY_PSEUDONYMIZE_USERS:
        return map_token(name_norm, 'USER', 'user_map', digits=3)
    return u



In [6]:
# --- 3) ZAT reader ---
log_reader = LogToDataFrame()

def read_zeek_log(path: Path) -> pd.DataFrame:
    df = log_reader.create_dataframe(str(path))
    # Standardize column names (zat already uses the #fields names)
    df.columns = [c.strip() for c in df.columns]
    return df


In [7]:
USER_COL_CANDIDATES = {
    "user", "username", "auth_user", "client_user", "server_user", "uid_user"
}

FQDN_COL_PATTERNS = [
    re.compile(r'(^|\.)query$'),
    re.compile(r'(^|\.)host$'),
    re.compile(r'(^|\.)server_name$'),
    re.compile(r'(^|\.)sni$'),
    re.compile(r'(^|\.)domain$'),
]

def pick_user_cols(cols):
    out = []
    for c in cols:
        if c.lower() in USER_COL_CANDIDATES:
            out.append(c)
    return out


def pick_fqdn_cols(cols):
    out = []
    for c in cols:
        for p in FQDN_COL_PATTERNS:
            if p.search(c.lower()):
                out.append(c)
                break
    return out

def clean_zeek_df(df: pd.DataFrame, log_type: str) -> pd.DataFrame:
    # Normalize placeholders
    for c in df.columns:
        df[c] = df[c].map(normalize_scalar)

    # User columns: newt-only
    for c in pick_user_cols(df.columns):
        df[c] = df[c].map(sanitize_user_token)

    # FQDN columns: Azure-only
    for c in pick_fqdn_cols(df.columns):
        df[c] = df[c].map(sanitize_fqdn)

    # Helpful standard cols
    df.insert(0, 'log_type', log_type)
    if 'ts' in df.columns:
        df['ts_utc'] = pd.to_datetime(df['ts'], unit='s', utc=True, errors='coerce')
    return df
 


In [8]:
# --- 5) Process all logs ---
results = {}

for sc, by_type in INPUTS.items():
    sc_out = OUT_DIR / sc
    sc_out.mkdir(parents=True, exist_ok=True)

    for log_type, paths in by_type.items():
        if not paths:
            continue

        # If multiple files per type, concatenate
        frames = []
        for p in sorted(paths):
            df = read_zeek_log(p)
            frames.append(df)
        df_all = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

        df_clean = clean_zeek_df(df_all, log_type)
        results[(sc, log_type)] = df_clean

        out_csv  = sc_out / f'zeek_{log_type}.csv'
        out_parq = sc_out / f'zeek_{log_type}.parquet'

        # If ts ended up as index, bring it back as a column
        if "ts" not in df_clean.columns and getattr(df_clean.index, "name", None) == "ts":
            df_clean = df_clean.reset_index()

        # Some readers may name it differently; normalize
        if "ts" not in df_clean.columns and "timestamp" in df_clean.columns:
            df_clean = df_clean.rename(columns={"timestamp": "ts"})
            
        df_clean.to_csv(out_csv, index=False)
        df_clean.to_parquet(out_parq, engine='fastparquet', index=False)
        print(f'[{sc}] {log_type}: wrote {out_csv.name}, {out_parq.name}')

print('\nDone. Cleaned:', len(results), 'scenario/log-type tables')


[sc7] conn: wrote zeek_conn.csv, zeek_conn.parquet
[sc7] dns: wrote zeek_dns.csv, zeek_dns.parquet
[sc7] http: wrote zeek_http.csv, zeek_http.parquet
[sc7] ssl: wrote zeek_ssl.csv, zeek_ssl.parquet
[sc7] files: wrote zeek_files.csv, zeek_files.parquet
[sc7] ssh: wrote zeek_ssh.csv, zeek_ssh.parquet
[sc3] conn: wrote zeek_conn.csv, zeek_conn.parquet
[sc3] dns: wrote zeek_dns.csv, zeek_dns.parquet
[sc3] http: wrote zeek_http.csv, zeek_http.parquet
[sc3] ssl: wrote zeek_ssl.csv, zeek_ssl.parquet
[sc3] files: wrote zeek_files.csv, zeek_files.parquet
[sc3] ssh: wrote zeek_ssh.csv, zeek_ssh.parquet
[sc4] conn: wrote zeek_conn.csv, zeek_conn.parquet
[sc4] dns: wrote zeek_dns.csv, zeek_dns.parquet
[sc4] http: wrote zeek_http.csv, zeek_http.parquet
[sc4] ssl: wrote zeek_ssl.csv, zeek_ssl.parquet
[sc4] files: wrote zeek_files.csv, zeek_files.parquet
[sc4] ssh: wrote zeek_ssh.csv, zeek_ssh.parquet

Done. Cleaned: 18 scenario/log-type tables


In [9]:
# --- 6) Quick peek ---
for lt, df in results.items():
    print("\n===", lt, "===")
    display(df.head(3))



=== ('sc7', 'conn') ===


,log_type,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,resp_bytes,conn_state,local_orig,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto
ts,,,,,,,,,,,,,,,,,,,,,,
2025-12-08 11:00:01.258867024,conn,CrFjkA18l0y1HwDSM7,10.0.0.4,43770,51.140.148.51,443,tcp,None,NaT,NaN,NaN,OTH,T,F,0,C,0,0,0,0,None,6
2025-12-08 10:59:57.969414949,conn,CDbDS33LPZYAvNMTv4,10.0.0.4,52660,169.254.169.254,80,tcp,None,0 days 00:00:00.008584,0.0,1535.0,SHR,T,T,0,^hCdfa,0,0,4,1751,None,6
2025-12-08 10:59:59.343611002,conn,CStgCc3qQtU0bCTVL1,10.0.0.4,48600,168.63.129.16,80,tcp,None,0 days 00:00:00.002196,0.0,2266.0,SHR,T,F,0,^hCdfa,0,0,4,2482,None,6



=== ('sc7', 'dns') ===


,log_type,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,trans_id,rtt,query,qclass,qclass_name,qtype,qtype_name,rcode,rcode_name,AA,TC,RD,RA,Z,answers,TTLs,rejected
ts,,,,,,,,,,,,,,,,,,,,,,,,
2025-12-08 11:00:00.618978977,dns,CRxDcotDLuz0kFNYe,10.0.0.4,42227,168.63.129.16,53,udp,3482,NaT,4.0.0.10.in-addr.arpa,None,None,None,None,0,NOERROR,F,F,F,T,0,dev-linux.internal.cloudapp.net,10.000000,F
2025-12-08 11:00:00.628201962,dns,CR3L5z1EFJ7Zjxgs77,10.0.0.4,53335,168.63.129.16,53,udp,5730,NaT,None,None,None,None,None,3,NXDOMAIN,F,F,F,F,0,None,None,F
2025-12-08 11:00:00.650248051,dns,CVPgS51dQPJSFyzvxh,10.0.0.4,35688,168.63.129.16,53,udp,21476,NaT,None,None,None,None,None,3,NXDOMAIN,F,F,F,F,0,None,None,F



=== ('sc7', 'http') ===


...

,log_type,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,version,cipher,curve,server_name,resumed,last_alert,next_protocol,established,ssl_history,cert_chain_fps,client_cert_chain_fps,sni_matches_cert,validation_status
ts,,,,,,,,,,,,,,,,,,,
2025-12-15 14:37:16.320724010,ssl,CpThqJ14sNjhOowyJ6,10.0.0.4,49720,185.199.108.153,443,TLSv13,TLS_AES_128_GCM_SHA256,x25519,None,F,None,None,F,si,None,None,None,None
2025-12-15 14:37:16.335570097,ssl,CfknZ913CgSPgYyU02,10.0.0.4,49888,185.199.109.153,443,TLSv13,TLS_AES_128_GCM_SHA256,x25519,None,F,None,None,F,si,None,None,None,None
2025-12-15 14:37:16.351051092,ssl,CV1hmMrvykXF1dxK8,10.0.0.4,57194,143.244.38.136,443,TLSv13,TLS_AES_256_GCM_SHA384,x25519,None,F,None,None,F,si,None,None,None,None



=== ('sc4', 'files') ===


,log_type,fuid,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,source,depth,analyzers,mime_type,filename,duration,local_orig,is_orig,seen_bytes,total_bytes,missing_bytes,overflow_bytes,timedout,parent_fuid,md5,sha1,sha256,extracted,extracted_cutoff,extracted_size
ts,,,,,,,,,,,,,,,,,,,,,,,,,,,
2025-12-15 14:36:34.070144892,files,F7pa903cn6yLKH4Ja8,Cantno1vOBp60kyZKf,10.0.0.4,55554,168.63.129.16,80,HTTP,0,"MD5,SHA1",application/xml,None,0 days 00:00:00.000007,F,F,2102,2102.0,0,0,F,None,4fe01f17bb8a2fcd2eb31ffe1538dae6,7b7f7b4d0a97506f9c8dbbeee88879dba838a8e5,None,None,None,None
2025-12-15 14:36:40.085875034,files,FIkhN52Sdo3IN6uhPj,Cmovzz1QoEakpdUjx3,10.0.0.4,55564,168.63.129.16,80,HTTP,0,"MD5,SHA1",application/xml,None,0 days 00:00:00.000004,F,F,2102,2102.0,0,0,F,None,4fe01f17bb8a2fcd2eb31ffe1538dae6,7b7f7b4d0a97506f9c8dbbeee88879dba838a8e5,None,None,None,None
2025-12-15 14:36:46.097342014,files,FMoQfc2KACSWd5iTBh,CEvrPF48mQcRpAFJM,10.0.0.4,38270,168.63.129.16,80,HTTP,0,"MD5,SHA1",application/xml,None,0 days 00:00:00.000003,F,F,2102,2102.0,0,0,F,None,4fe01f17bb8a2fcd2eb31ffe1538dae6,7b7f7b4d0a97506f9c8dbbeee88879dba838a8e5,None,None,None,None



=== ('sc4', 'ssh') ===


,log_type,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,version,auth_success,auth_attempts,direction,client,server,cipher_alg,mac_alg,compression_alg,kex_alg,host_key_alg,host_key,remote_location.country_code,remote_location.region,remote_location.city,remote_location.latitude,remote_location.longitude
ts,,,,,,,,,,,,,,,,,,,,,,,
2025-12-15 14:40:41.306639910,ssh,CHJSQs0Bweig300Qk,80.94.95.116,63946,10.0.0.4,22,None,None,0,INBOUND,SSH-2.0-libssh_0.10.5,None,None,None,None,None,None,None,None,None,None,None,None
2025-12-15 14:42:15.911676884,ssh,CF8s2g3c80qkwog8sb,193.32.162.146,37240,10.0.0.4,22,None,None,0,INBOUND,SSH-2.0-Go,None,None,None,None,None,None,None,None,None,None,None,None
2025-12-15 14:45:25.954459906,ssh,Cp3lnQIMU8dlcSGGk,193.32.162.146,49398,10.0.0.4,22,None,None,0,INBOUND,SSH-2.0-Go,None,None,None,None,None,None,None,None,None,None,None,None
